In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')

Database config loaded: localhost
Connected to old database: dataleap_v5_example_new
Connected to new database: dataleap_v5_migration
Connected to future database: dataleap_v5_migration


In [3]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 3 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_3_data = {}

# 1. Load File Cimut
try:
    with open('fase_3_cimut.pkl', 'rb') as f:
        all_fase_3_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_3_afrida.pkl', 'rb') as f:
        all_fase_3_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_3_hanif.pkl'):
        with open('fase_3_hanif.pkl', 'rb') as f:
            all_fase_3_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    print(f"⚠️ Gagal memuat file pkl Hanif: {f}")
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 3 (SALING SILANG & AUTO-SKIP) 🚀 


✓ Berhasil memuat data hasil konversi Cimut.
✓ Berhasil memuat data hasil konversi Afrida.
✓ Berhasil memuat data hasil konversi Hanif.


In [4]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Susunan di bawah ini diatur ketat lintas personel agar Foreign Key aman masuk ke MySQL!
tables_to_insert_ordered_1= [
    # --- BLOK A: PENDAFTARAN & SDM (Karya Hanif) ---
    'pelamar',                  # Induk data pelamar kerja/kursus
    'pelamar_kerja',            # Detail pelamar posisi kerja
    'pelamar_sekolah',          # Riwayat sekolah pelamar
    'pelamar_kursus',           # Riwayat kursus pelamar
    'progres_pelamar',          # Log catatan tahapan seleksi
    'rekrutmen_pelamar',        # Keputusan akhir rekrutmen pelamar
    'pengajuan_karyawan',       # Form pengajuan penambahan staff baru
    'histori_pengajuan',        # Log alur persetujuan pengajuan staff
]

In [5]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Susunan di bawah ini diatur ketat lintas personel agar Foreign Key aman masuk ke MySQL!
tables_to_insert_ordered_2 = [
    # --- BLOK B: SURAT-MENYURAT & SOP (Karya Afrida) ---
    'sop',                      # Standar operasional prosedur instansi
    'surat_keluar',             # Log keluar dokumen/surat resmi
    'verifikasi_surat_keluar',  # Log persetujuan surat keluar oleh atasan
    'surat_tugas',              # Surat perintah penugasan formal
    'surat_tugas_anggota',      # Anggota staff yang terikat di dalam surat tugas
    'sop_kategori', 
]

In [6]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Susunan di bawah ini diatur ketat lintas personel agar Foreign Key aman masuk ke MySQL!
tables_to_insert_ordered_3 = [
    # --- BLOK C: MARKETING & ADMISI CALON SISWA (Karya Cimut) ---
    'kontak_prospek',           # Database mentah leads / prospek marketing
    'calon_siswa',              # Formulir profil utama calon siswa baru
    'calon_siswa_ortu',         # Data wali / orang tua calon siswa
    'calon_siswa_akademik',     # Riwayat background akademik calon siswa
    'calon_siswa_bayar',        # Log transaksi pembayaran formulir/DP awal
    'calon_siswa_jadwal',       # Plotting jadwal tes/interview calon siswa
    'calon_siswa_kursus',       # Pilihan program kursus yang diminati calon siswa
    'calon_siswa_proses',       # Jalur perkembangan dokumen admisinya
    'calon_siswa_status_logs',  # Log perubahan status akhir (Diterima/Ditolak/Pending)

    # --- BLOK D: LOGISTIK & OPERASIONAL INTERNAL (Karya Cimut) ---
    'pengadaan',                # Form pengajuan belanja/pengadaan aset barang
    'peminjaman',               # Log pinjam pakai sarana prasarana oleh staff
    'problem'                   # Log laporan kerusakan/kendala teknis fasilitas
]

In [7]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name]) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name])
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [8]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_3 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=all_fase_3_data, 
    ordered_list=tables_to_insert_ordered_1
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)



 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ pelamar: Sukses diproses! Sebanyak 192 baris sukses dimasukkan / di-skip aman.
  ✓ pelamar_kerja: Sukses diproses! Sebanyak 67 baris sukses dimasukkan / di-skip aman.
  ✓ pelamar_sekolah: Sukses diproses! Sebanyak 53 baris sukses dimasukkan / di-skip aman.
  ✓ pelamar_kursus: Sukses diproses! Sebanyak 50 baris sukses dimasukkan / di-skip aman.
  ✓ progres_pelamar: Sukses diproses! Sebanyak 403 baris sukses dimasukkan / di-skip aman.
  ✓ rekrutmen_pelamar: Sukses diproses! Sebanyak 281 baris sukses dimasukkan / di-skip aman.
  ✓ pengajuan_karyawan: Sukses diproses! Sebanyak 33 baris sukses dimasukkan / di-skip aman.
  ✓ histori_pengajuan: Sukses diproses! Sebanyak 79 baris sukses dimasukkan / di-skip aman.

🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):
  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.

📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL

📂 [🟢 PREVIEW TABEL SUKSE

,id_pengajuan,email_pelamar,nama_lengkap,nama_panggilan,jenis_kelamin,tempat_lahir,tanggal_lahir,alamat_ktp,alamat_domisili,nomor_wa,akun_linkedin,akun_instagram,akun_facebook,sosmed_lain,spesifikasi_laptop,internet,kegiatan_sekarang,rencana_karir,mobilitas,sumber_info,siap_wfo,tanggal_bergabung,kategori_pelamar,riwayat_kerja,riwayat_pendidikan,pengalaman_bidang,wawasan,riwayat_kesehatan,status_pernikahan,kemampuan_ajar,penguasaan_aplikasi,aplikasi_lainnya,penggunaan_laptop,skor_toefl,ekspektasi_gaji,tautan_berkas,alasan_resign,skor_iq,foto_iq,foto_minat,foto_kepribadian,created_at
0,<NA>,ditari@leapsurabaya.sch.id,-,-,Perempuan,-,1970-01-01,-,-,-,NaN,-,-,-,-,-,-,-,-,-,-,1970-01-01,Part Time English Teacher,-,-,-,-,-,Belum Menikah,-,-,-,Tidak Pernah,0,0.0,-,-,0,-,-,-,2020-01-01 00:00:00
1,<NA>,hartikaharahap95@gmail.com,Hartika Prawidaningrum Harahap,Tika,Perempuan,Sidoarjo,1970-01-01,Perum Grand Surya Cluster Jupiter Blok D12/16 ...,Perum Grand Surya Cluster Jupiter Blok D12/16 ...,'089678227530,-,-,-,-,"Ya, Punya",WiFi,mengurus rumah tangga,Bekerja,Motor Pribadi,Saudara atau kerabat,bersedia,2023-06-21,Admin Prakerja,"01 September 2018 –30 April 2023 PT. Bank BSI,...",2012 – 2016 STIE Perbanas Surabaya,<p>1. 20 Juni 2016 &ndash; 8 Desember 2017</p>...,-,-,Belum Menikah,Tidak Pernah,"Microsoft Office (Word, Power Point, dll);,Zoo...",tidak pernah,Pernah,507,4550000.0,https://drive.google.com/open?id=1Z_FpilagwmNd...,sedang tidak bekerja,100,1688095776_3de967eefe836d28e873.jpeg,1688095993_b75d248ce60436d0d4a1.jpg,1688096006_bf7b1c0e082c6aaf5e67.jpeg,2023-06-29 10:24:53
2,<NA>,admin@gmail.com,sdasd,sadas,Perempuan,asd,1970-01-01,asd,asda,532453,sadas,,,-,Tidak Punya,WiFi,ZX,ZX,Motor Pribadi,LinkedIn,-,1970-01-01,Coding,sad,asd,<p>asdasd</p>,asda,asd,Belum Menikah,Ya Pernah,Zoom Meeting;,ZX,Tidak Pernah,0,0.0,-,-,0,-,-,-,2020-01-01 00:00:00
3,<NA>,nirmalapradnyas@gmail.com,Ni Putu Jayanti Nirmala Pradnya Santosa,Nirmala,Perempuan,Surabaya,1970-01-01,Bendul Merisi Permai blok C no 22 Surabaya,Surabaya,082131446266,-,https://www.instagram.com/nrimlaa/,-,-,"Ya, Punya",WiFi,Menyelesaikan skripsi,Saya berencana segera menyelesaikan kuliah sam...,Motor Pribadi,IG Leap Surabaya,-,1970-01-01,,Mahasiswi,SMA,"<p style=""padding-left: 30px;"">Saya belum memi...",Lembaga belajar nonformal yang membantu anak a...,Maag,Belum Menikah,Tidak Pernah,"Microsoft Office (Word, Power Point, dll);,Goo...",-,Tidak Pernah,517,0.0,-,-,0,-,-,-,2020-01-01 00:00:00
4,2,nirmalapradnyas@gmail.com,Ni Putu Jayanti Nirmala Pradnya Santosa,Nirmala,Perempuan,Surabaya,1970-01-01,Bendul Merisi Permai blok C no 22 Surabaya,Surabaya,082131446266,-,https://www.instagram.com/nrimlaa/,-,-,"Ya, Punya",WiFi,Saya merupakan mahasiswi semester akhir yang s...,"Rencana saya, segera menyelesaikan pendidikan ...",Motor Pribadi,IG Leap Surabaya,"Ya, interaksi offline menyenangkan & dapat ilmu",2023-07-10,Marketing,Mahasiswi,SMA,<p>Saya belum pernah bekerja ataupun magang se...,Leap merupakan lembaga bimbingan belajar profe...,Maag,Belum Menikah,Tidak Pernah,"Microsoft Office (Word, Power Point, dll);,Goo...",-,Pernah,517,0.0,https://drive.google.com/file/d/1u8_ZrepEz7mEG...,-,94,1688617417_0765b3a992194874930b.png,1688617538_60b8cc4bd0dd8da5c0b4.jpg,1688617547_8eb3154d5d4cb7210165.png,2023-07-05 11:17:01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,<NA>,siti.uswatun@leapsurabaya.sch.id,Siti Uswatun Khasanah,-,Perempuan,-,1970-01-01,-,-,-,NaN,-,-,-,-,-,-,-,-,-,-,1970-01-01,-,-,-,-,-,-,Belum Menikah,-,-,-,Tidak Pernah,0,0.0,-,-,0,-,-,-,2020-01-01 00:00:00
188,<NA>,rini.rahayu@leapsurabaya.sch.id,"Rini Budi Rahayu, S.Pd.",-,Perempuan,-,1970-01-01,-,-,-,NaN,-,-,-,-,-,-,-,-,-,-,1970-01-01,-,-,-,-,-,-,Belum Menikah,-,-,-,Tidak Pernah,0,0.0,-,-,0,-,-,-,2020-01-01 00:00:00
189,<NA>,vivi.wulandari@leapsurabaya.sch.id,Vivi Pramitha Wulandari,-,Perempuan,-,1970-

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PELAMAR_KERJA]
--------------------------------------------------


,id_pelamar,nama_perusahaan,periode,jabatan,deskripsi_kerja
0,101,Coding Bee Academy,2021-2022,Educator,<p>- Membuat lesson plan</p>\r\n<p>- Membuat s...
1,181,Pusat Bahasa UINSA Surabaya,2011 - sampai sekarang,Tutor Bahasa Inggris,<p>Mengajar dua kelas pada semester 1 dan 2. D...
2,182,PT Aku Pintar Indonesia,2019-2021,Tutor Team Lead dan English Tutor,"<p><span style=""color: rgba(0, 0, 0, 0.9); fon..."
3,150,INFOMEDIA NUSANTARA,2018-2020,CALL CENTER BNI,<p>Melayani keluhan dan kebutuhan pelanggan BN...
4,183,LKP LEAP English & Digital Surabaya,2022-Sekarang,Part time pengajar Bahasa Inggris,<p>Mengajar Siswa</p>
...,...,...,...,...,...
62,138,The Ritz-Carlton Bali,12 Agustus 2024 – 12 Februari 2025,Food & Beverage Service Intern,"<p style=""text-align: justify;"">Memberikan lay..."
63,138,Kampus Mengajar (Kemendikbud),14 Agustus – 1 Desember 2023,Teaching Assistant,"<p style=""text-align: justify;"">Membantu guru ..."
64,138,Sproutgigs.com,14 Januari 2022 – 1 Juli 2024,Freelance Data Entry,"<p style=""text-align: justify;"">Memasukkan dat..."
65,160,Sunshine Learning Center,Juli 2025 - Desember 2025,Part time English Teacher,<p>Bertanggung jawab untuk mendukung proses be...


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PENGAJUAN_KARYAWAN]
--------------------------------------------------


,id_user,posisi,jumlah,syarat,pertanyaan,alur_seleksi,daftar_tes,status,created_at
0,U00016,Part-Time Offline English Teacher,2,"<h5 class=""t-20 mb3"" style=""box-sizing: border...",<p>Info Jam Kerja dan Gaji :</p>\r\n<p>&nbsp;<...,<p>1. Mengisi form dan test melalui link: http...,<p>Mempersiapkan bahan micro teaching (MT) onl...,Diterima,2023-06-16 17:02:47
1,U00014,Magang Sales & Marketing,1,<p>1. Background pendidikan apa saja</p>\r\n<p...,<p>1. Komitmen kapan bisa mulai dan lama magan...,<p>1. Seleksi administrasi</p>\r\n<p>2. Probin...,<p>1. Buatlah desain poster sederhana program ...,Diterima,2023-07-04 13:52:25
2,U00014,Volunteer LeapXperience,1,<p>1. Mahasiswa dari berbagai jurusan (tingkat...,<p>1. Apakah bisa hadir offline ke Leap setiap...,<p>Seleksi Administrasi &amp; Skill :</p>\r\n<...,<p>Tes sudah terintegrasi dalam tahap administ...,Diterima,2023-07-18 10:11:14
3,U00012,Karyawan IT Support serta GA,1,<p>- Pendidikan minimal SMK atau Sarjana (S1) ...,<p>1. Berikan contoh pengalamanmu dalam menyel...,<p>Tahap 1: Pengumuman lowongan dan penerimaan...,<p>Pertanyaan ini mencakup 10 pertanyaan esai<...,Diterima,2023-07-18 11:24:24
4,U00016,Instruktur Aplikasi Perkantoran,2,"<p><span class=""selectable-text copyable-text""...","<p class=""selectable-text copyable-text iq0m55...",<p>Tahap 1: Pengumuman lowongan dan penerimaan...,<p>detail test menyusul dan didiskusikan</p>,Diterima,2023-07-18 16:36:22
5,U00012,Freelance Guru Coding Scratch,1,<p>1. Penguasaan bahasa pemrograman Scratch se...,<p>Apakah kamu bisa menjelaskan beberapa blok ...,"<p style=""box-sizing: border-box; margin-top: ...",<p>Waktu Tes (60 Menit)<br />Soal 1 :<br />Bua...,Diterima,2023-07-20 13:59:46
6,U00012,Part Time Teacher,1,<p>- Menjadi lulusan S1 atau sedang menempuh s...,<p>1. Bagaimana kamu akan menunjukkan semangat...,"<p><span style=""box-sizing: border-box; color:...","<p><span style=""color: #212529; font-family: R...",Diterima,2023-08-08 16:55:25
7,U00020,Freelance TK Mitra,3,<p>wanita</p>\r\n<p>mahasiswa/ lulusan S1 bhs ...,"<p dir=""ltr"" style=""line-height: 1.2; margin-t...",<p>1. probing</p>\r\n<p>2. isi form lamar</p>\...,<p>....</p>,Diterima,2023-08-28 14:40:03
8,U00014,Freelance Desainer Grafis Edtech Conference,1,<p>1. Background pendidikan tidak dibatasi</p>...,<p>1. Apa saja pengalaman dalam membuat desain...,<p>1. Seleksi Administrasi</p>\r\n<p>2. Probin...,<p>1. Membuat 1 desain poster promosi Leap sec...,Diterima,2023-09-21 15:40:36
9,U00014,Freelance Artikel (SEO),1,<p>1. All backgrounds</p>\r\n<p>2. Berpengalam...,<p>1. Sudah berpengalaman berapa lama menulis ...,<p>1. Probing</p>\r\n<p>2. Seleksi admininstra...,<p>1. Tulis 1 buah artikel maksimal 1000 kata ...,Diterima,2023-10-13 17:00:40


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: HISTORI_PENGAJUAN]
--------------------------------------------------


,id_pengajuan,status_verifikasi_pengajuan,catatan,created_at
0,1,Diajukan,NaN,2023-06-16 17:02:47
1,2,Diajukan,NaN,2023-07-04 13:52:25
2,2,Diterima,"Mbak, mohon diinfokan untuk tes ini harus dila...",2023-07-04 14:13:49
3,3,Diajukan,NaN,2023-07-18 10:11:14
4,3,Revisi,"Mbak Laksmi, ini kemarin infonya dibutuhkan 2 ...",2023-07-18 10:21:19
...,...,...,...,...
74,32,Diajukan,NaN,2025-11-06 15:24:10
75,32,Revisi,,2025-11-06 15:41:54
76,32,Sudah Revisi,NaN,2025-11-06 15:42:26
77,32,Diterima,,2025-11-06 15:42:41


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [9]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_3 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=all_fase_3_data, 
    ordered_list=tables_to_insert_ordered_2
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ sop: Sukses diproses! Sebanyak 4 baris sukses dimasukkan / di-skip aman.
  ✓ surat_keluar: Sukses diproses! Sebanyak 231 baris sukses dimasukkan / di-skip aman.
  ✓ verifikasi_surat_keluar: Sukses diproses! Sebanyak 513 baris sukses dimasukkan / di-skip aman.
  ✓ surat_tugas: Sukses diproses! Sebanyak 139 baris sukses dimasukkan / di-skip aman.
  ✓ surat_tugas_anggota: Sukses diproses! Sebanyak 319 baris sukses dimasukkan / di-skip aman.
  ✓ sop_kategori: Sukses diproses! Sebanyak 2 baris sukses dimasukkan / di-skip aman.

🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):
  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.

📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL

📂 [🟢 PREVIEW TABEL SUKSES: SOP]
--------------------------------------------------


,judul_sop,link_dokumen_sop,created_at,id_sop_kategori
0,Akhir Penggunaan Kelas English (19.15 WIB),https://drive.google.com/file/d/1V0MpB7ctzPHe6...,2023-07-07 08:52:48,1
1,Penerimaan Surat Masuk,https://drive.google.com/file/d/1G8lO33yU7MufC...,2023-11-22 15:27:06,2
2,Pengajuan Surat Keluar,https://drive.google.com/file/d/14I7wYSk62WJAv...,2023-11-22 15:27:55,2
3,Pengajuan Surat Tugas,https://drive.google.com/file/d/1R-wKFqQeVSz62...,2023-11-22 15:28:27,2


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: SURAT_KELUAR]
--------------------------------------------------


,id_sk,id_user,keterangan_sk,link_dokumen_sk,status_sk,nomor_sk,catatan_sk,created_at
0,29,U00011,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,Disetujui,105/LEAP/BD/XI/2023,Tidak ada catatan,2023-11-13 15:41:58
1,27,U00011,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,Disetujui,102/LEAP/BD/X/2023,Tidak ada catatan,2023-10-26 17:37:59
2,28,U00026,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,Disetujui,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,Tidak ada catatan,2023-11-10 16:10:04
3,26,U00011,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,Disetujui,101/LEAP/BD/X/2023,Tidak ada catatan,2023-10-25 16:57:38
4,24,U00011,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,Sudah Revisi,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...,2023-09-11 17:57:49
...,...,...,...,...,...,...,...,...
226,268,U00034,Surat Izin Uji Coba Proyek SMPN 52,https://docs.google.com/document/d/1XEh8HQTo8F...,Disetujui,070/PDDK/PM/LEAP/VI/2026,Tidak ada catatan,2026-06-02 10:15:56
227,269,U00033,Pelaporan Hasil Ujian Semester Genap Kelas 1-5...,https://docs.google.com/document/d/13mJO4GDm_6...,Disetujui,039/PDDK/SKET/MIM/VI/2026,Tidak ada catatan,2026-06-04 15:39:50
228,270,U00034,Surat Ijin Tdk Menghadiri Kegiatan Belajar Agnes,https://docs.google.com/document/d/1t6qRBEClQW...,Disetujui,072/PDDK/SI/LEAP/VI/2026,Tidak ada catatan,2026-06-08 10:56:13
229,271,U00023,MoM Diskusi Placement Test Batch 2,https://docs.google.com/document/d/1m3zumPebn5...,NaN,Nomor surat belum diisi,Tidak ada catatan,2026-06-10 13:55:27


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: VERIFIKASI_SURAT_KELUAR]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: SURAT_TUGAS_ANGGOTA]
--------------------------------------------------


,id_st,id_user
0,6,U00012
1,6,U00003
2,7,U00026
3,7,U00012
4,8,U00026
...,...,...
314,170,U00035
315,170,U00052
316,170,U00070
317,171,U00014


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: SOP_KATEGORI]
--------------------------------------------------


,id_sop_kategori,nama_kategori_sop
0,1,Kelas
1,2,HR / GA


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [10]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_3 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=all_fase_3_data, 
    ordered_list=tables_to_insert_ordered_3
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)



 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ kontak_prospek: Sukses diproses! Sebanyak 194 baris sukses dimasukkan / di-skip aman.
  ✓ calon_siswa_kursus: Sukses diproses! Sebanyak 166 baris sukses dimasukkan / di-skip aman.
  ✓ pengadaan: Sukses diproses! Sebanyak 110 baris sukses dimasukkan / di-skip aman.
  ✓ peminjaman: Sukses diproses! Sebanyak 194 baris sukses dimasukkan / di-skip aman.
  ✓ problem: Sukses diproses! Sebanyak 160 baris sukses dimasukkan / di-skip aman.

🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):
  ✗ calon_siswa: Gagal total saat insert - Alasan: 1054 (42S22): Unknown column 'fo_status' in 'field list'
  ✗ calon_siswa_ortu: Gagal total saat insert - Alasan: 1054 (42S22): Unknown column 'tempat_lahir_ayah' in 'field list'
  ✗ calon_siswa_akademik: Gagal total saat insert - Alasan: 1054 (42S22): Unknown column 'submission_state' in 'field list'
  ✗ calon_siswa_bayar: Gagal total saat insert - Alasan: 1054 (4

,id_kontak_prospek,kode_kontak,nama_penanya,nomor_telepon,email,sumber_informasi,catatan_awal_fo,id_admin_fo,status_kontak,tanggal_kontak_pertama,tanggal_kontak_terakhir
0,1,PIJWREZC,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,Teman/kerabat/saudara,None,None,joined,None,2025-09-18 17:39:40
1,2,LEHUEB6B,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,Instagram,None,None,joined,None,2025-09-23 16:25:14
2,3,HBJ6O8TF,Zulfa Bariatur Rahma,085707179656,chyzryth@gmail.com,Lainnya,None,None,joined,None,2025-10-03 09:48:34
3,4,LI0QW5C1,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,Instagram,None,None,joined,None,2025-10-03 09:49:15
4,5,CEFRJE2H,Khansa Amalia Putri Aji,081554932188,afadhilpa@gmail.com,Teman/kerabat/saudara,None,None,in_disscussion,None,2025-10-03 09:48:50
...,...,...,...,...,...,...,...,...,...,...,...
189,190,8F37BXJC,Atiya Sabita,-,None,None,None,None,waiting for confirmation,None,2025-10-17 15:41:09
190,191,CXYHPRFZ,Dwi,-,None,None,None,None,follow up another time,None,2025-10-17 15:41:26
191,192,9WBVZDEI,Aca,-,None,None,None,None,follow up another time,None,2025-10-24 15:27:54
192,193,6GPQ5V44,Bu Hartik,-,None,None,None,None,waiting for confirmation,None,2025-10-24 15:28:21


--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: CALON_SISWA] 🚨
Alasan MySQL Menolak: ✗ calon_siswa: Gagal total saat insert - Alasan: 1054 (42S22): Unknown column 'fo_status' in 'field list'
--------------------------------------------------
Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):


,id_calon,kode_unik,nama_lengkap,id_kontak_prospek,nama_panggilan,jenis_kelamin,tempat_lahir,tanggal_lahir,kewarganegaraan,email,agama,nama_kontak_awal,wa_kontak_awal,id_provinsi,id_kabupaten,id_kecamatan,id_kelurahan,alamat_lengkap,wa_siswa,wa_ortu,wa_administrasi,assigned_fo,assigned_akademik,catatan_awal_fo,fo_status,fo_status_updated_at,handover_at,link_form_sent_at,form_completed_at,first_submitted_at,latest_submitted_at,deleted_at,created_at,updated_at
0,1,PIJWREZC,Daria Azmiya Jasmine,1,Daria,Perempuan,None,None,Indonesia,puterihapsari.f@gmail.com,None,Ibu Sari,082231346758,11.0,162.0,None,None,None,None,082231346758,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,2025-09-18 17:39:40
1,2,LEHUEB6B,Annisa Zahro Ramadhania,2,Annisa,Perempuan,None,None,Indonesia,dwirohm4@gmail.com,None,Bu Dwi,081336647476,11.0,162.0,None,None,None,None,081336647476,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,2025-09-23 16:25:14
2,3,HBJ6O8TF,Zulfa Bariatur Rahma,3,Zulfa,Perempuan,None,None,Indonesia,chyzryth@gmail.com,None,Ibu Fitri,083849241708,11.0,162.0,None,None,None,083849241708,085707179656,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,2025-10-03 09:48:34
3,4,LI0QW5C1,Achmad Naufal Albiruni,4,Albi,None,None,None,Indonesia,melisnifuku@gmail.com,None,Bu Lita,087765283592,11.0,162.0,None,None,None,None,087765283592,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,2025-10-03 09:49:15
4,5,CEFRJE2H,Khansa Amalia Putri Aji,5,Khansa,Perempuan,None,None,Indonesia,afadhilpa@gmail.com,None,Ibu Fadhil,081554932188,11.0,162.0,None,None,None,081554932188,081554932188,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,2025-10-03 09:48:50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,162,1H85H846,Antonius Miguel Kurniawan,162,Miguel,None,None,None,Indonesia,adeodatus.kurniawan@gmail.com,None,None,None,11.0,162.0,None,None,None,None,08170326911,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,None
162,163,CWWQZ9HU,Tisha kayla janitra,163,Tisha,Perempuan,None,None,Indonesia,nroskalindha18@gmail.com,None,None,None,11.0,162.0,None,None,None,None,082244441630,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,None
163,164,ROLCHKDI,Clariza Arifianti,164,Clara,Perempuan,None,None,Indonesia,clarizarisa3@gmail.com,None,None,None,11.0,162.0,None,None,None,None,08977257033,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,None
164,165,WVUWAQMD,Shaqila Anindra Dzakira,165,Shaqila,Perempuan,None,None,Indonesia,shaqilaanindra@gmail.com,None,None,None,11.0,162.0,None,None,None,None,085850209079,None,None,None,None,new_lead,None,None,None,None,None,None,None,None,None


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PEMINJAMAN]
--------------------------------------------------


,id_pinjam,tanggal_pinjam,keperluan,id_user,status_pinjam,catatan_sarpras,created_at
0,2,2023-06-11,<p>pinjam kamera - fun class tk mitra - 1 - 11...,U00026,Selesai,,2023-06-12 13:20:39
1,3,2023-06-11,<p>1. kamera - fun class TK mitra - 1 - 11 Jun...,U00026,Selesai,,2023-06-12 13:21:35
2,5,2023-08-21,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Mid Te...,U00026,Selesai,,2023-08-16 15:01:55
3,6,2023-08-23,<p>Pinjam kamera untuk rekaman video checklist...,U00033,Selesai,,2023-08-23 10:19:12
4,7,2023-10-11,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Final ...,U00026,Selesai,,2023-10-10 15:30:53
...,...,...,...,...,...,...,...
189,192,2026-04-10,"<p><span style=""color: #212529; font-family: R...",U00060,Selesai,<p>Sudah dikembalikan</p>,2026-04-09 11:40:52
190,193,2026-04-10,"<p><span style=""color: #212529; font-family: R...",U00041,Disetujui,,2026-04-10 16:26:19
191,194,2026-04-17,<p>List Peminjaman barang kegiatan student app...,U00060,Disetujui,,2026-04-15 16:06:08
192,195,2026-04-17,"<p><span style=""color: #212529; font-family: R...",U00060,Disetujui,,2026-04-16 15:22:28


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PROBLEM]
--------------------------------------------------


,id_problem,detail_masalah,id_user,status_perbaikan,tanggal_lapor,tanggal_selesai,catatan_teknisi,gambar_problem
0,43,AC Kelas Miss Erika kurang dingin ( belakang,U00012,Proses,2023-06-28 16:07:52,NaT,"sudah info ke Pak Irawan,\r\nTukang AC masih l...",None
1,58,Boya/mic untuk kelas hybrid tidak berfungsi (,U00026,Terselesaikan,2023-07-06 10:45:03,2023-07-17,,None
2,60,tegangan listrik di ruang kelas belakang dapur...,U00033,Terselesaikan,2023-07-13 16:59:57,2023-08-10,pemberian stabilizer,None
3,61,ac brisik,U00033,Terselesaikan,2023-07-14 09:24:29,2023-07-25,sudah tidak berisik\r\n,None
4,62,Kabel power monitor PC room 2 longgar. Saat me...,U00036,Terselesaikan,2023-07-17 15:04:01,2023-07-17,,None
...,...,...,...,...,...,...,...,...
155,251,"AC room 6 tidak dingin, dan ada air menetes da...",U00050,Terselesaikan,2026-04-08 16:55:36,2026-04-13,,None
156,252,AC Ruang 6 (miss Peni) kondisi saat ini di OFF...,U00020,Terselesaikan,2026-04-09 16:31:44,2026-04-13,AC sudah diperbaiki,/storage/sarpas_images/sarpas_69d772008efe9.jpeg
157,253,Sesi 3. Hybrid. Guru menggunakan mic unt hybri...,U00026,Terselesaikan,2026-04-13 19:02:53,2026-04-15,,/storage/sarpas_images/sarpas_69dcdb6dd7663.jpg
158,254,"Bracket TV kurang kenceng, suka geser kedepan ...",U00019,Terselesaikan,2026-04-16 16:58:21,2026-04-17,,None


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [11]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 3 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_3 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )